In [8]:
%pip install -U deepeval


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from pathlib import Path
import pandas as pd
from deepeval import evaluate
from deepeval.metrics import MCPUseMetric
from deepeval.test_case import LLMTestCase, MCPServer, MCPToolCall
from deepeval.models import DeepSeekModel, AnthropicModel
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.sse import sse_client
from contextlib import AsyncExitStack

In [10]:
# --- DeepEval JSON robustness patch (Anthropic) ---
# DeepEval 3.8.4 expects the evaluation model to output a single JSON object.
# Some Anthropic responses can contain multiple JSON blocks, causing JSONDecodeError: Extra data.
# This patch extracts *all* JSON objects and prefers the one that looks like a metric output.

import json
import re
from deepeval.errors import DeepEvalError
import deepeval.models.llms.utils as deepeval_llm_utils
import deepeval.models.llms.anthropic_model as deepeval_anthropic

_orig_trim_and_load_json = deepeval_llm_utils.trim_and_load_json

def robust_trim_and_load_json(input_string: str):
    # Strip common Markdown fences before attempting JSON decoding.
    cleaned = re.sub(r"```(?:json)?", "", input_string, flags=re.IGNORECASE)
    cleaned = cleaned.replace("```", "")

    decoder = json.JSONDecoder()
    found_objects: list[dict] = []
    scan_index = 0

    while True:
        start = cleaned.find("{", scan_index)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(cleaned[start:])  # parses a single JSON value
            if isinstance(obj, dict):
                found_objects.append(obj)
            scan_index = start + end
        except json.JSONDecodeError:
            scan_index = start + 1

    if not found_objects:
        # Fall back to DeepEval's original heuristic parser (will raise DeepEvalError if invalid).
        return _orig_trim_and_load_json(input_string)

    # Prefer the object that looks like DeepEval metric output.
    for obj in reversed(found_objects):
        if "score" in obj and "reason" in obj:
            return obj

    # Otherwise, return the last parsed JSON object.
    return found_objects[-1]

# Patch both the utils module and the Anthropic wrapper (which imports the function directly).
deepeval_llm_utils.trim_and_load_json = robust_trim_and_load_json
deepeval_anthropic.trim_and_load_json = robust_trim_and_load_json
print("Applied robust JSON parsing patch for DeepEval AnthropicModel")

Applied robust JSON parsing patch for DeepEval AnthropicModel


In [11]:
load_dotenv()  
# api_key = os.getenv("DEEPSEEK_API_KEY")

# model = DeepSeekModel(
#     model="deepseek-chat",
#     api_key=api_key
# )

api_key = os.getenv("ANTHROPIC_API_KEY")

model = AnthropicModel(
    model="claude-opus-4-6",
    api_key=api_key,
    temperature=0,
 )

metric = MCPUseMetric(model=model)


In [12]:
async def describe_mcp_server(url: str, server_name: str = "orion-mcp-server") -> MCPServer:
    """Connect to the MCP server and capture the primitives it exposes."""
    exit_stack = AsyncExitStack()
    try:
        read_stream, write_stream = await exit_stack.enter_async_context(
            sse_client(url)
        )
        session = await exit_stack.enter_async_context(
            ClientSession(read_stream, write_stream)
        )
        await session.initialize()

        tool_list = await session.list_tools()
        try:
            resource_list = await session.list_resources()
        except AttributeError:
            resource_list = None
        try:
            prompt_list = await session.list_prompts()
        except AttributeError:
            prompt_list = None

        return MCPServer(
            server_name=server_name,
            transport="sse",
            available_tools=tool_list.tools,
            available_resources=resource_list.resources if resource_list else None,
            available_prompts=prompt_list.prompts if prompt_list else None,
        )
    finally:
        await exit_stack.aclose()

try:
    mcp_server = await describe_mcp_server("http://mcp-server.smo:8000/sse")
except Exception as exc:
    mcp_server = None
    print(f"Failed to connect to MCP server: {exc}")

In [13]:
initial_prompt = (
    """
        You are an assistant that manages network slice reservations for developers via MCP tool calls.
        Follow these rules:
        1. The number of devices (UEs), service time, and service area are not required values.
        2. The number of devices does not have a defined maximum value, as it depends on the value entered by the user.
        3. Consider the number of UEs/devices only when defined by the user.
        4. Ask for throughput/downstream/upstream units only when the user provides a value without any unit. If the value already includes a unit (accept case-insensitive variants such as bps, kbps, Mbps, Gbps, Tbps, Mb/s, megabits per second, etc.), proceed without asking again and reuse that unit.
        5. Service time and area stay null unless the user specifies them.
        6. If a latency/delay budget is given without a unit, assume "Milliseconds".
        7. Don't treat mentions of cost or budget as guidance for choosing conservative values, only populate fields that the user requested.
        8. All fields stay null unless the user specifies them, never fabricate values.
    """
)

In [14]:
csv_path = Path("/home/vmadmin/intent/src/test/validated_outputs/anthropic-sonnet-4-5.csv")
raw_df = pd.read_csv(csv_path)
#raw_df = raw_df.head(2)
raw_df = raw_df.fillna(0)
raw_df

,intent,intent_processing,tool_call,type_definition,policy
0,"Provision a slice for a TV broadcast van zone,...","Message(id='msg_01EQLAboYRLca7dqgJ67v7TB', con...","meta=None content=[TextContent(type='text', te...","Message(id='msg_01CkLRCBa63YFhujDXHuduT8', con...","{""status"":""Policy created successfully"",""code""..."
1,"Establish a slice for co-working spaces, suppo...","Message(id='msg_017CbNhbkYJcr8yDsPS5K5WS', con...","meta=None content=[TextContent(type='text', te...","Message(id='msg_01U7ucECRGrV978tmeHTewtF', con...","{""status"":""Policy created successfully"",""code""..."
2,Create a slice for a retail analytics platform...,0,"Message(id='msg_01PquyGKCxyHj7zFPRyc96KB', con...",0,0
3,Deploy a slice for mobile gaming users in a ci...,0,"Message(id='msg_012JWc2JUSnemXpXVstBsh2x', con...",0,0
4,Provision a slice for financial traders on the...,0,"Message(id='msg_015gHwVBhTq4VnFKtkGkMFM6', con...",0,0
...,...,...,...,...,...
80,Provision a slice for smart greenhouse climate...,0,"Message(id='msg_017gjb2A76nRPWEvkrG8Y8gf', con...",0,0
81,Establish a slice for public safety siren moni...,0,"Message(id='msg_015jHpFaEEzFyHCzTV1ursn9', con...",0,0
82,Deploy a slice for high-resolution mobile mapp...,0,"Message(id='msg_01LsSZhPyQ4GsShNGsoHC5Rm', con...",0,0
83,Create a slice for autonomous ferry navigation...,0,"Message(id='msg_01KPzj5fSFy5mMQTDwNRXoJ3', con...",0,0


In [15]:
result_rows = []

In [16]:
print(result_rows)

[]


In [ ]:
for index, row in raw_df.iterrows():
    output = ""
    if row.intent_processing != 0:
        output = str(row.intent_processing)
    else:
        output = str(row.tool_call)
    
    messages=[{"role": "user", "content": initial_prompt}]
    user_intent = {"role": "user", "content": str(row.intent)}
    messages.append(user_intent)
    test_case = LLMTestCase(
        input=str(messages),
        actual_output=output,
        mcp_servers=[mcp_server],
    )

    metric.measure(test_case)
    result_rows.append({
        "success": bool(metric.success),
        "score": float(metric.score) if metric.score is not None else None,
        "reason": str(metric.reason) if metric.reason is not None else None,
    })

results_df = pd.DataFrame(result_rows, columns=["success", "score", "reason"])
out_path = Path("/home/vmadmin/intent/src/results/deepeval_sonnet-4-5_2.csv")
results_df.to_csv(out_path, index=False)
print(f"Saved results to: {out_path}")
results_df